In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Geração de embeddings com Sentence Transformers (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)

In [ ]:
import gc
from pathlib import Path

import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

MERGED = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged"
)

EMBEDDINGS = MERGED / "embeddings"
EMBEDDINGS.mkdir(parents=True, exist_ok=True)

ENTRADA = MERGED / "banco_python_final.parquet"

print("Carregando banco...")

banco = pd.read_parquet(ENTRADA)

print(f"Total de registros: {len(banco):,}")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo: {device}")

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=device
)

BATCH_SIZE = 5000
ENCODE_BATCH_SIZE = 32

for inicio in tqdm(
    range(0, len(banco), BATCH_SIZE),
    desc="Gerando embeddings"
):

    fim = min(inicio + BATCH_SIZE, len(banco))

    arquivo_saida = EMBEDDINGS / f"lote_{inicio:07d}.parquet"

    if arquivo_saida.exists():
        continue

    try:

        lote = banco.iloc[inicio:fim].copy()

        textos = (
            "Question:\n"
            + lote["question"].fillna("")
            + "\n\nAnswer:\n"
            + lote["answer"].fillna("")
        ).tolist()

        embeddings = model.encode(
            textos,
            batch_size=ENCODE_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        embeddings = embeddings.astype("float32")

        lote["embedding"] = [
            emb for emb in embeddings
        ]

        lote.to_parquet(
            arquivo_saida,
            index=False
        )

    except Exception as e:

        print(f"\nErro no lote {inicio}: {e}")

    finally:

        del lote
        del textos

        if "embeddings" in locals():
            del embeddings

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nTodos os lotes foram processados!")

Confirmação da geração de embeddings em lotes

In [ ]:
from pathlib import Path

EMBEDDINGS = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings"
)

arquivos = list(EMBEDDINGS.glob("*.parquet"))

print("Arquivos encontrados:", len(arquivos))

for arquivo in arquivos[:5]:
    print(arquivo.name)

Validação de um lote

In [ ]:
import pandas as pd
from pathlib import Path

arquivo = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings/lote_0000000.parquet"
)

df = pd.read_parquet(arquivo)

print(df.shape)
print(df.columns)

print("\nPrimeiro registro:")
print(df.iloc[0][["question", "answer"]])

print("\nDimensão do embedding:")
print(len(df.iloc[0]["embedding"]))

Verificação do assuntos e estrutura dos lotes

In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings/lote_0320000.parquet"
]

for file in files:
    print("=" * 100)
    print(f"Arquivo: {file}")

    df = pd.read_parquet(file)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nPrimeiras linhas:")

    print(df.head())

Conversão de embeddings de float64 para float32

In [ ]:
import gc
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.auto import tqdm


EMBEDDINGS = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings"
)

arquivos = sorted(EMBEDDINGS.glob("*.parquet"))

print("Total de lotes:", len(arquivos))


for arquivo in tqdm(
    arquivos,
    desc="Convertendo embeddings"
):

    temporario = arquivo.with_suffix(".tmp.parquet")

    try:

        if temporario.exists():
            temporario.unlink()


        df = pd.read_parquet(arquivo)


        dtype_atual = df["embedding"].iloc[0].dtype


        if dtype_atual == np.float32:
            print(f"{arquivo.name} já está em float32")
            continue

        df["embedding"] = df["embedding"].apply(
            lambda x: np.asarray(
                x,
                dtype=np.float32
            )
        )


        df.to_parquet(
            temporario,
            index=False,
            engine="pyarrow",
            compression="snappy"
        )

        temporario.replace(arquivo)


        print(f"{arquivo.name} convertido")


    except Exception as e:

        print(
            f"\nErro no lote {arquivo.name}: {e}"
        )


        if temporario.exists():
            temporario.unlink()


    finally:

        if "df" in locals():
            del df

        gc.collect()


print("\nConversão finalizada!")

Verificação da conversão em um lote de embeddings

In [ ]:
df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings/lote_0150000.parquet"
)

print(df["embedding"].iloc[0].dtype)
print(df["embedding"].iloc[0].shape)

Validação da conversão de lotes

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

EMBEDDINGS = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings"
)

erros = []

for arquivo in sorted(EMBEDDINGS.glob("*.parquet")):
    df = pd.read_parquet(arquivo)

    dtype = df["embedding"].iloc[0].dtype

    if dtype != np.float32:
        erros.append(
            (arquivo.name, dtype)
        )

print("Lotes com problema:", len(erros))

if erros:
    print(erros[:10])